# 05 · The PyTorch Workflow and Best Practices

**Part II — Training deep networks** · Estimated time: ~120 min

Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization was about *what* to do when training a network: which optimizer, which schedule, which initialization, which regularizer. This notebook is about *how* to do it well in PyTorch: how to write models as `nn.Module`s that are easy to configure and inspect, how to feed data with `Dataset` and `DataLoader`, how to write a training loop that you can reuse for every project in this course, and how to log, checkpoint, reproduce and debug a run. The centrepiece is a complete FashionMNIST project, an MLP and a small convolutional network trained to about 90% and 92% test accuracy, with TensorBoard logging, checkpointing, a confusion matrix and error analysis. Everything we build here is reused by the later notebooks, and the final section explains how to move it out of a notebook into a real project.

## Learning objectives

By the end of this notebook you will be able to:

- Write custom `nn.Module`s with configurable architectures, and explain the difference between parameters, buffers and submodules, and what `state_dict` contains.
- Implement `Dataset` classes for tabular data and image folders, use `DataLoader` with shuffling, custom collation and samplers, and split data correctly with different train and evaluation transforms.
- Build reusable `train_one_epoch`, `evaluate` and `fit` functions with learning-rate scheduling, gradient clipping, early stopping, checkpointing and a history dictionary.
- Log scalars, images, histograms and the model graph to TensorBoard and keep a configuration file with every run.
- Save and load complete checkpoints (model, optimizer, scheduler, epoch, RNG state) and resume training exactly where it stopped.
- Seed every source of randomness, explain what determinism PyTorch can and cannot guarantee, and report results properly.
- Train and evaluate a complete FashionMNIST classifier with augmentation, confusion matrix, per-class accuracy and error analysis.
- Debug shape errors, `nan`s and silent bugs with assertions, anomaly detection, hooks and the overfit-one-batch test.

## Prerequisites

- Notebook 03 · Tensors and Autograd (tensors, autograd, the preview of `nn.Module` and `torch.optim`).
- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (optimizers, schedules, BatchNorm, dropout, early stopping, reading loss curves).

## Contents

1. `nn.Module` anatomy
2. Data: `Dataset` and `DataLoader`
3. The training loop, done properly
4. Logging and experiment tracking with TensorBoard
5. Checkpointing and resuming
6. Reproducibility
7. A complete project: FashionMNIST
8. Debugging deep learning code
9. Project structure for real work

Exercises · Summary · Further reading

In [ ]:
import json
import math, os, random, time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset, random_split, WeightedRandomSampler
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.utils import make_grid
from PIL import Image, ImageDraw
from sklearn.metrics import confusion_matrix, classification_report

# Must be set before the first CUDA matrix multiply if Section 6 is to make cuBLAS deterministic.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = DATA_DIR / "runs"            # TensorBoard event files
CKPT_DIR = DATA_DIR / "checkpoints"     # saved models
RUNS_DIR.mkdir(exist_ok=True); CKPT_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Optional libraries: the notebook runs without them, with reduced functionality.
try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False
    print("tensorboard is not installed (see environment.yml); a no-op SummaryWriter will be used.")

    class SummaryWriter:
        """Stand-in that accepts every call and does nothing."""
        def __init__(self, *args, **kwargs):
            pass

        def __getattr__(self, name):
            return lambda *args, **kwargs: None

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except ImportError:
    HAS_TORCHINFO = False
    print("torchinfo is not installed (see environment.yml); model summaries will be skipped.")

try:
    import yaml
    HAS_YAML = True
except ImportError:
    HAS_YAML = False
    print("pyyaml is not installed; configs will be saved as JSON only.")

## 1. `nn.Module` anatomy

### 1.1 `__init__` and `forward`

Every model, layer and loss in PyTorch is an `nn.Module`. The contract is small: in `__init__` you create the pieces (layers, parameters, constants), and in `forward` you describe how an input flows through them. Everything else, the bookkeeping of parameters, moving to a device, saving and loading, switching between training and evaluation mode, hooks, is inherited. Three rules cover most of what can go wrong:

1. Call `super().__init__()` first; it sets up the internal registries that make the rest work.
2. Store layers as attributes (`self.fc = nn.Linear(...)`). Assigning an `nn.Module` or an `nn.Parameter` to an attribute *registers* it, which is how `model.parameters()` finds it.
3. Never call `forward` directly; call the module, `model(x)`. `__call__` runs the forward pass together with the forward and backward hooks (Section 8.3), which a direct `forward` call skips.

Below is a configurable MLP written this way. It takes images as $(B, 1, 28, 28)$ tensors and flattens them itself, so that the data pipeline does not need to know which model consumes it. Its hidden part is an `nn.Sequential` built from a tuple of widths, which is the usual way to make an architecture configurable from a dictionary of hyperparameters. The second model is a small convolutional network for the same task; convolutions are the subject of Notebook 06 · Convolutional Neural Networks, and here it is enough to read `features` as "a learned feature extractor that turns a $1\times 28\times 28$ image into a $64\times 7\times 7$ block of features".

In [ ]:
class MLP(nn.Module):
    """Configurable multi-layer perceptron for (B, 1, 28, 28) images or (B, 784) vectors."""
    def __init__(self, in_dim=784, hidden_dims=(512, 256), n_classes=10, dropout=0.0, activation=nn.ReLU):
        super().__init__()
        dims = [in_dim, *hidden_dims]
        layers = []
        for d_in, d_out in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(d_in, d_out), activation()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.hidden = nn.Sequential(*layers)
        self.head = nn.Linear(dims[-1], n_classes)

    def forward(self, x):
        x = x.flatten(start_dim=1)            # (B, 1, 28, 28) -> (B, 784); leaves (B, 784) unchanged
        return self.head(self.hidden(x))


class SmallCNN(nn.Module):
    """Two conv blocks (conv - BatchNorm - ReLU - max-pool) and a small classifier; ~420k parameters."""
    def __init__(self, n_classes=10, channels=(32, 64), dropout=0.3):
        super().__init__()
        c1, c2 = channels
        self.features = nn.Sequential(
            nn.Conv2d(1, c1, kernel_size=3, padding=1), nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),                                        # 28x28 -> 14x14
            nn.Conv2d(c1, c2, kernel_size=3, padding=1), nn.BatchNorm2d(c2), nn.ReLU(),
            nn.MaxPool2d(2),                                        # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(c2 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        assert x.ndim == 4 and tuple(x.shape[1:]) == (1, 28, 28), f"expected (B, 1, 28, 28), got {tuple(x.shape)}"
        return self.classifier(self.features(x))


model = MLP(hidden_dims=(256, 128), dropout=0.2).to(device)
x = torch.randn(4, 1, 28, 28, device=device)
print(model)
print("output shape:", tuple(model(x).shape))

Printing a module shows its tree of submodules, which is a quick check that the architecture is what you intended. The output has shape $(4, 10)$: one row of ten **logits** per image. No softmax: as in Notebooks 02–04, the softmax is folded into `F.cross_entropy`, and the model outputs raw scores.

### 1.2 Parameters, buffers and `state_dict`

A module holds three kinds of state. **Parameters** (`nn.Parameter`) are tensors that the optimizer updates; `nn.Linear` creates `weight` and `bias`. **Buffers** are tensors that belong to the model but are not trained: BatchNorm's running mean and variance are the classic example. **Submodules** are other modules, which recursively hold their own parameters and buffers. `model.state_dict()` is a dictionary of *all* parameters and buffers, keyed by their dotted path in the module tree, and is what gets saved to disk. Buffers are registered with `register_buffer`; they are moved by `.to(device)` and saved with the model (unless registered with `persistent=False`), but `parameters()` does not return them, so the optimizer never touches them.

The cell below defines a small `Standardize` module whose mean and standard deviation are buffers, which is a neat way of shipping the preprocessing statistics *inside* the model (Exercise 1 uses it for a regression task), and inspects what is registered where.

In [ ]:
class Standardize(nn.Module):
    """(x - mean) / std with the statistics stored as buffers: saved with the model, moved by .to(), never trained."""
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32))
        self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32))

    def forward(self, x):
        return (x - self.mean) / self.std


demo = nn.Sequential(Standardize([0.5, 1.0], [2.0, 4.0]), nn.Linear(2, 3), nn.BatchNorm1d(3)).to(device)
print("state_dict keys:")
for k, v in demo.state_dict().items():
    print(f"   {k:24s} shape {tuple(v.shape)}  ({'parameter' if k in dict(demo.named_parameters()) else 'buffer'})")
print("named_parameters:", [name for name, _ in demo.named_parameters()])
print("named_buffers:   ", [name for name, _ in demo.named_buffers()])
print("device of the buffer after .to(device):", demo[0].mean.device)

The `state_dict` contains both parameters and buffers (BatchNorm's `num_batches_tracked` counter is a buffer too), whereas `named_parameters()` lists only the trainable tensors. The keys are the attribute paths: `1.weight` is the weight of the second element of the `Sequential`. When you write a custom module, a tensor attribute that is neither a `Parameter` nor a registered buffer is *invisible* to all of this machinery: it will not be saved, and `.to(device)` will not move it, which produces a "tensors on different devices" error on the GPU. Use `register_buffer` for such constants (or `nn.Parameter(..., requires_grad=False)`).

### 1.3 Containers: `nn.Sequential`, `nn.ModuleList`, `nn.ModuleDict`

Submodules are registered only when they are assigned as attributes or stored in one of PyTorch's container modules. A plain Python list of layers is *not* registered, and this is one of the most common silent bugs: the model runs, produces outputs, but the layers in the list never receive gradient updates because `parameters()` does not know about them. `nn.ModuleList` behaves like a list but registers its contents; `nn.ModuleDict` does the same for a dictionary (useful for multiple heads); `nn.Sequential` additionally defines `forward` as the chain of its children.

In [ ]:
class BrokenStack(nn.Module):
    def __init__(self, depth=3, width=8):
        super().__init__()
        self.layers = [nn.Linear(width, width) for _ in range(depth)]      # plain list: NOT registered

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x


class FixedStack(nn.Module):
    def __init__(self, depth=3, width=8):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(width, width) for _ in range(depth)])

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x


class MultiHead(nn.Module):
    """A shared body with a dictionary of task-specific heads."""
    def __init__(self, in_dim=8, tasks=None):
        super().__init__()
        tasks = tasks or {"digit": 10, "parity": 2}
        self.body = nn.Sequential(nn.Linear(in_dim, 16), nn.ReLU())
        self.heads = nn.ModuleDict({name: nn.Linear(16, n_out) for name, n_out in tasks.items()})

    def forward(self, x, task):
        return self.heads[task](self.body(x))


for cls in (BrokenStack, FixedStack, MultiHead):
    m = cls()
    print(f"{cls.__name__:12s}: {sum(p.numel() for p in m.parameters()):5d} parameters visible to the optimizer")
print("state_dict of BrokenStack:", list(BrokenStack().state_dict().keys()))

`BrokenStack` reports zero parameters and an empty `state_dict`: an optimizer built from `BrokenStack().parameters()` would train nothing, and the model would forget its weights when saved. The `ModuleList` version registers all three layers. Always compare the parameter count with what you expect.

### 1.4 Initialization with `apply`, summaries and parameter counts

`module.apply(fn)` calls `fn` on every submodule (children first, then the module itself), which is the idiomatic way to re-initialize weights by layer type, as we did in Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization. `torchinfo.summary` prints the layer-by-layer output shapes and parameter counts for a given input size, which is the fastest way to catch a shape mistake in a new architecture. Finally, `model.train()` and `model.eval()` set a `training` flag on the module and recursively on all its children, which is what makes `nn.Dropout` and `nn.BatchNorm` change behaviour.

In [ ]:
def init_weights(m):
    """He init for conv/linear weights, zero biases, BatchNorm at (1, 0)."""
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)


def count_parameters(model, trainable_only=True):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


cnn = SmallCNN().to(device)
cnn.apply(init_weights)
print(f"SmallCNN: {count_parameters(cnn):,} trainable parameters; MLP(512, 256): {count_parameters(MLP()):,}")
cnn.eval()
print("training flags after cnn.eval():", {name: m.training for name, m in cnn.named_modules() if isinstance(m, (nn.Dropout, nn.BatchNorm2d))})
cnn.train()
if HAS_TORCHINFO:
    print(torchinfo_summary(cnn, input_size=(1, 1, 28, 28), device=str(device), verbose=0))
else:
    print("(torchinfo not available: install it to see the per-layer summary)")

The summary shows how the $28\times 28$ image shrinks to $7\times 7$ while the channel count grows, and that almost all of the CNN's parameters sit in the first fully-connected layer, which is typical and is why modern architectures replace it with global average pooling (Notebook 06 · Convolutional Neural Networks).

## 2. Data: `Dataset` and `DataLoader`

### 2.1 The `Dataset` protocol

`torch.utils.data.Dataset` is an interface with two methods: `__len__` returns the number of examples, and `__getitem__(i)` returns example `i`, usually as a tuple `(input, target)` of tensors. That is all a `DataLoader` needs; where the data lives (memory, a CSV file, image files, a database, a remote bucket) and how it is preprocessed are entirely up to you. The "map-style" dataset is by far the most common; there is also `IterableDataset` for streams whose length is unknown, which we do not need here.

Our first dataset reads a CSV-like table. We synthesize one with NumPy, write it into `DATA_DIR` (so that the example is self-contained), and read it back. Loading everything into memory in `__init__` is the right choice for a table that fits; `__getitem__` then just indexes.

In [ ]:
TABLE_PATH = DATA_DIR / "synthetic_table.csv"
rng = np.random.default_rng(SEED)
features = rng.normal(size=(500, 5)) * np.array([1.0, 5.0, 0.1, 2.0, 1.0]) + np.array([0.0, 10.0, 0.0, -3.0, 1.0])
labels = (features[:, 0] + 0.5 * features[:, 3] + rng.normal(scale=0.5, size=500) > 0).astype(int)
np.savetxt(TABLE_PATH, np.column_stack([features, labels]), delimiter=",", fmt="%.5f",
           header="f0,f1,f2,f3,f4,label", comments="")


class TableDataset(Dataset):
    """A CSV table with the label in the last column; the whole file is loaded into two tensors."""
    def __init__(self, path):
        arr = np.genfromtxt(path, delimiter=",", skip_header=1)
        self.X = torch.tensor(arr[:, :-1], dtype=torch.float32)
        self.y = torch.tensor(arr[:, -1], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


table = TableDataset(TABLE_PATH)
x0, y0 = table[0]
print(f"{len(table)} rows; example 0: x = {x0.cpu().numpy().round(2)}, y = {y0.item()}")
print(f"class balance: {table.y.float().mean().item():.2f} positives")

### 2.2 `DataLoader`: batching, shuffling, collation, samplers

A `DataLoader` turns a dataset into an iterator over mini-batches. Its job has three parts. A **sampler** decides the order of indices (`SequentialSampler` when `shuffle=False`, `RandomSampler` when `shuffle=True`, or any sampler you pass); a **batch sampler** groups them into lists of `batch_size` indices (with `drop_last=True` discarding the incomplete final batch, which matters for BatchNorm and for schedulers that count steps); and a **collate function** turns the list of examples into a batch. The default `collate_fn` stacks tensors along a new first dimension, and it recurses into tuples and dictionaries, which is why `(x, y)` examples come out as `(X_batch, y_batch)`.

In [ ]:
loader = DataLoader(table, batch_size=64, shuffle=True, drop_last=False,
                    generator=torch.Generator().manual_seed(SEED))
xb, yb = next(iter(loader))
print(f"one batch: x {tuple(xb.shape)} {xb.dtype}, y {tuple(yb.shape)} {yb.dtype}; "
      f"{len(loader)} batches per epoch (last one has {len(table) % 64} rows)")
print("with drop_last=True:", len(DataLoader(table, batch_size=64, shuffle=True, drop_last=True)), "batches")
print("first indices without shuffling:", list(iter(DataLoader(table, batch_size=5, shuffle=False).sampler))[:5])
print("first indices with shuffling:   ", list(iter(loader.sampler))[:5])

The default collation fails as soon as examples have different shapes, which is the everyday situation with text (sentences of different length) and with variable-size images. The fix is a custom `collate_fn` that pads the examples to a common size and returns whatever extra information the model needs, such as the original lengths. The next cell shows the failure and the fix on a dataset of random integer sequences; Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs uses exactly this pattern with `pack_padded_sequence`.

In [ ]:
class RandomSequences(Dataset):
    """Integer sequences of random length 3..max_len (token 0 is reserved for padding); label = parity of the length."""
    def __init__(self, n=40, max_len=9, vocab=20, seed=SEED):
        g = torch.Generator().manual_seed(seed)
        lengths = torch.randint(3, max_len + 1, (n,), generator=g)
        self.seqs = [torch.randint(1, vocab, (int(L),), generator=g) for L in lengths]
        self.labels = [int(L) % 2 for L in lengths]

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        return self.seqs[i], torch.tensor(self.labels[i])


def pad_collate(batch):
    """Pad sequences to the longest in the batch; return (padded, lengths, labels)."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, lengths, torch.stack(labels)


seq_ds = RandomSequences()
try:
    next(iter(DataLoader(seq_ds, batch_size=4)))
except RuntimeError as e:
    print("default collate fails:", str(e).splitlines()[0][:90])
else:
    print("(the first four sequences happened to have equal lengths, so the default collate succeeded this time)")
padded, lengths, labels = next(iter(DataLoader(seq_ds, batch_size=4, collate_fn=pad_collate)))
print(f"custom collate: padded batch {tuple(padded.shape)}, lengths {lengths.tolist()}, labels {labels.tolist()}")
print(padded)

### 2.3 Images in folders

The most common layout for image classification data is one sub-folder per class, `root/class_name/image.png`. `torchvision.datasets.ImageFolder` reads it directly, assigning class indices in alphabetical order of the folder names. To see what it does, we generate a tiny synthetic dataset of circles and squares on the fly with Pillow (guarded so that it is only created once), load it with `ImageFolder`, and then write our own equivalent `Dataset` in a few lines: scan the folders, keep a list of `(path, label)` pairs, and open and transform the image lazily in `__getitem__`. Lazy loading is the point: a folder of a million JPEGs cannot be loaded into memory in `__init__`, but a list of a million paths can.

In [ ]:
SHAPES_DIR = DATA_DIR / "synthetic_shapes"


def make_shape_image(kind, rng, size=32):
    img = Image.new("L", (size, size), color=0)
    draw = ImageDraw.Draw(img)
    r = int(rng.integers(5, 10))
    cx, cy = (int(v) for v in rng.integers(r + 1, size - r - 1, size=2))
    box = [cx - r, cy - r, cx + r, cy + r]
    if kind == "circle":
        draw.ellipse(box, fill=int(rng.integers(150, 256)))
    else:
        draw.rectangle(box, fill=int(rng.integers(150, 256)))
    return img


if not SHAPES_DIR.exists():
    rng = np.random.default_rng(SEED)
    for kind in ("circle", "square"):
        (SHAPES_DIR / kind).mkdir(parents=True)
        for k in range(20):
            make_shape_image(kind, rng).save(SHAPES_DIR / kind / f"{kind}_{k:02d}.png")
print("files per class:", {d.name: len(list(d.glob("*.png"))) for d in sorted(SHAPES_DIR.iterdir())})

shape_tf = v2.Compose([v2.Grayscale(num_output_channels=1), v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
folder_ds = datasets.ImageFolder(str(SHAPES_DIR), transform=shape_tf)


class FolderImageDataset(Dataset):
    """Minimal re-implementation of ImageFolder: class = sub-folder name, indices in sorted order, lazy loading."""
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.classes = sorted(p.name for p in self.root.iterdir() if p.is_dir())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = [(p, self.class_to_idx[c]) for c in self.classes for p in sorted((self.root / c).glob("*.png"))]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("L")           # open lazily; "L" = 8-bit grey-scale
        if self.transform is not None:
            img = self.transform(img)
        return img, label


ours_ds = FolderImageDataset(SHAPES_DIR, transform=shape_tf)
img_a, lab_a = folder_ds[0]
img_b, lab_b = ours_ds[0]
print(f"ImageFolder: classes {folder_ds.classes}, {len(folder_ds)} images, item 0 -> {tuple(img_a.shape)} {img_a.dtype}, label {lab_a}")
print(f"ours:        classes {ours_ds.classes}, {len(ours_ds)} images, identical first item: {torch.equal(img_a, img_b) and lab_a == lab_b}")
grid = make_grid(torch.stack([ours_ds[i][0] for i in [0, 1, 2, 3, 20, 21, 22, 23]]), nrow=8, pad_value=0.5)
plt.figure(figsize=(8, 1.8)); plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray"); plt.axis("off")
plt.title("synthetic shapes dataset: four circles, four squares"); plt.show()

### 2.4 `torchvision` datasets, `transforms.v2`, and splitting data correctly

`torchvision.datasets` provides the standard benchmarks with `download=True`, and `torchvision.transforms.v2` provides composable preprocessing and augmentation that works on PIL images, tensors and the `tv_tensors` types (images, bounding boxes, masks) alike. The pipeline for FashionMNIST is: convert the PIL image to a `uint8` image tensor (`ToImage`), optionally augment (`RandomCrop` with padding, `RandomHorizontalFlip`), convert to `float32` in $[0, 1]$ (`ToDtype(..., scale=True)`), and standardize with the dataset mean and standard deviation (`Normalize`). We compute the two statistics from the raw training images rather than copying them from a tutorial.

Now the split. We want a training set, a validation set for decisions (early stopping, hyperparameters) and the untouched test set. `random_split` is the one-liner for splitting a dataset, but it has a subtlety: the two halves are `Subset` views of the *same* dataset object and therefore share its `transform`, so you cannot augment the training half without also augmenting the validation half. The robust pattern is to create **two dataset objects** with different transforms (augmenting and plain) and split by **indices**, using `Subset` with a fixed random permutation. Under `FAST_RUN` we use 6,000 training and 1,000 validation images; the full run uses 55,000 and 5,000.

In [ ]:
fm_raw = datasets.FashionMNIST(str(DATA_DIR), train=True, download=True)
CLASSES = fm_raw.classes
FM_MEAN = (fm_raw.data.float() / 255).mean().item()
FM_STD = (fm_raw.data.float() / 255).std().item()
print(f"FashionMNIST: {len(fm_raw)} training images, mean {FM_MEAN:.4f}, std {FM_STD:.4f}")

eval_tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize([FM_MEAN], [FM_STD])])
train_tf = v2.Compose([v2.ToImage(), v2.RandomCrop(28, padding=2), v2.RandomHorizontalFlip(),
                       v2.ToDtype(torch.float32, scale=True), v2.Normalize([FM_MEAN], [FM_STD])])

fm_train_aug = datasets.FashionMNIST(str(DATA_DIR), train=True, download=True, transform=train_tf)
fm_train_plain = datasets.FashionMNIST(str(DATA_DIR), train=True, download=True, transform=eval_tf)
fm_test = datasets.FashionMNIST(str(DATA_DIR), train=False, download=True, transform=eval_tf)

# random_split: fine when both halves should get the same transform
half_a, half_b = random_split(fm_train_plain, [55_000, 5_000], generator=torch.Generator().manual_seed(SEED))
print(f"random_split: {len(half_a)} + {len(half_b)}; both halves share transform: {half_a.dataset is half_b.dataset}")

# index split with two dataset objects: augmentation for training only
N_TRAIN = 6_000 if FAST_RUN else 55_000
N_VAL = 1_000 if FAST_RUN else 5_000
perm = torch.randperm(len(fm_raw), generator=torch.Generator().manual_seed(SEED)).tolist()
train_idx, val_idx = perm[:N_TRAIN], perm[N_TRAIN:N_TRAIN + N_VAL]
train_set = Subset(fm_train_aug, train_idx)          # augmented
train_set_plain = Subset(fm_train_plain, train_idx)  # same images, no augmentation (used for comparisons)
val_set = Subset(fm_train_plain, val_idx)
test_set = fm_test

BATCH_SIZE = 128
PIN = (device.type == "cuda")
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_set, batch_size=512, shuffle=False, num_workers=0, pin_memory=PIN)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=0, pin_memory=PIN)
print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)} images; "
      f"{len(train_loader)} training batches of {BATCH_SIZE}")

xb, yb = next(iter(train_loader))
print(f"a training batch: x {tuple(xb.shape)} {xb.dtype}, mean {xb.mean():.3f}, std {xb.std():.3f}; y {tuple(yb.shape)} {yb.dtype}")
imgs = torch.stack([train_set[0][0] for _ in range(8)] + [train_set_plain[0][0]])   # 8 augmented views + the original
grid = make_grid(imgs * FM_STD + FM_MEAN, nrow=9, pad_value=1.0)
plt.figure(figsize=(9, 1.8)); plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray"); plt.axis("off")
plt.title(f"eight augmented views of one training image (last: original), class '{CLASSES[train_set[0][1]]}'"); plt.show()

Each call to `train_set[0]` applies a fresh random crop and flip, so the eight views differ, while the plain dataset returns the original. Note the standardized batch statistics (mean near $0$, standard deviation near $1$), and that the labels are `int64` (`torch.long`), which is what `F.cross_entropy` requires for class indices.

### 2.5 Imbalanced classes: `WeightedRandomSampler`

When one class is rare, uniform shuffling produces batches in which the rare class barely appears, and the network learns to ignore it. A `WeightedRandomSampler` draws indices with replacement according to per-example weights; setting each example's weight to the inverse frequency of its class makes every class equally likely per batch. It is one of several remedies (others are class-weighted losses via `F.cross_entropy(weight=...)`, oversampling, and threshold tuning), and the right one depends on what you optimize; the sampler is the easiest to bolt onto an existing pipeline.

In [ ]:
# an imbalanced version of the table dataset: keep only 5% of class 1 (own generator, so the subset is the same on every run)
rng_imb = np.random.default_rng(SEED + 1)
keep = [i for i in range(len(table)) if table.y[i] == 0 or rng_imb.random() < 0.05]
imb = Subset(table, keep)
y_imb = table.y[keep]
counts = torch.bincount(y_imb, minlength=2)
print(f"imbalanced subset: {counts.tolist()} examples per class")

weights = (1.0 / counts.float())[y_imb]                       # weight of each example = 1 / (count of its class)
sampler = WeightedRandomSampler(weights, num_samples=len(imb), replacement=True,
                                generator=torch.Generator().manual_seed(SEED))
for name, ld in [("shuffle=True", DataLoader(imb, batch_size=32, shuffle=True)),
                 ("WeightedRandomSampler", DataLoader(imb, batch_size=32, sampler=sampler))]:
    fracs = [yb.float().mean().item() for _, yb in ld]
    print(f"{name:22s}: fraction of class 1 per batch, mean {np.mean(fracs):.2f}, min {min(fracs):.2f}")

With plain shuffling the handful of positives is spread over a dozen or so batches, so most batches contain none at all (the minimum fraction is $0$); with the sampler about half of each batch is positive. Two details: `sampler` and `shuffle=True` are mutually exclusive (the sampler defines the order), and with `replacement=True` an epoch is no longer "each example once", so the notion of an epoch becomes a convention (`num_samples` draws).

### 2.6 Data pipeline performance

The `DataLoader` is where a surprising fraction of training time is spent, because the per-example work (decoding a JPEG, augmenting, converting) runs in Python. The knobs, in order of importance:

- **`num_workers`**: number of subprocesses that prepare batches in parallel while the GPU computes. On Linux, $4$–$8$ workers are typical. On Windows (and macOS) workers are started with the `spawn` method, so the dataset, sampler and collate function must be picklable and importable by a fresh interpreter; classes defined inside a notebook usually are not, and the workers then fail or hang, which is why this course uses `num_workers=0` everywhere. In a script (Section 9), guarded by `if __name__ == "__main__":`, you can and should raise it. `persistent_workers=True` keeps the workers alive between epochs.
- **`pin_memory=True`** allocates batches in page-locked host memory, which makes the copy to the GPU faster and lets `.to(device, non_blocking=True)` overlap with computation. Harmless on CPU, hence our `pin_memory=(device.type == "cuda")`.
- **Do less per example**: decode once and cache small datasets as tensors (`TensorDataset`), store preprocessed arrays in `.npy`/`.pt` files, or move the augmentation to the GPU with batched `v2` transforms on tensors.
- **Measure**: if the GPU utilization is low and the epoch time does not change when you halve the model, the loader is the bottleneck. Notebook 14 · Efficient Training and Scaling profiles this properly.

A quick illustration of the third point: in the cell below the same 6,000-image pass is timed with the per-image transform pipeline and with a pre-tensorized `TensorDataset`.

In [ ]:
DEMO_N = 6_000
demo_idx = train_idx[:DEMO_N]
t0 = time.time()
for _ in DataLoader(Subset(fm_train_plain, demo_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0):
    pass
t_transform = time.time() - t0

X_cached = torch.stack([fm_train_plain[i][0] for i in demo_idx])     # tensorize once
y_cached = torch.tensor([fm_train_plain[i][1] for i in demo_idx])
t0 = time.time()
for _ in DataLoader(TensorDataset(X_cached, y_cached), batch_size=BATCH_SIZE, shuffle=False):
    pass
t_cached = time.time() - t0
print(f"one pass over {len(demo_idx)} images with per-image v2 transforms: {t_transform:.2f}s; "
      f"over the same images as cached tensors: {t_cached:.2f}s")

The transform pipeline is many times slower than iterating over cached tensors, and that gap is pure Python overhead. For small datasets like this one, caching the plain tensors and doing the augmentation on the batch (as in Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization, Section 6.5) is the fastest option; for large image datasets, worker processes are the answer.

## 3. The training loop, done properly

### 3.1 `train_one_epoch` and `evaluate`

The training loop in Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization was a function that worked on in-memory tensors. The version we write now works with any `DataLoader`, any model, any loss, and it is split into two functions with clear contracts:

- `train_one_epoch` makes one pass over the loader in training mode. For each batch it moves the data to the device, computes the loss, checks that it is finite (a `nan` should stop the run immediately with a clear error rather than silently corrupting the weights), zeroes the gradients, backpropagates, optionally clips the gradient norm, steps the optimizer and the scheduler, and accumulates the loss and a metric. It returns the epoch's mean loss and metric.
- `evaluate` makes one pass in evaluation mode under `torch.inference_mode()`, a stricter and faster cousin of `torch.no_grad()` that also disables version counters and view tracking, and returns the mean loss and metric.

Two details deserve a comment. We accumulate `loss.item() * batch_size` and divide by the number of examples at the end, so that the epoch loss is the exact mean over examples even when the last batch is smaller. And the metric is passed in as a function that returns a *sum* over the batch (number of correct predictions for accuracy), for the same reason; for a regression task you pass a different function (Exercise 1) or `None`.

In [ ]:
def accuracy_sum(outputs, targets):
    """Number of correct predictions in a batch (a sum, so batches of different sizes average correctly)."""
    return (outputs.argmax(dim=1) == targets).sum().item()


def train_one_epoch(model, loader, optimizer, loss_fn=F.cross_entropy, scheduler=None, clip_norm=None,
                    metric_fn=accuracy_sum, writer=None, step_offset=0):
    """One pass over `loader` in training mode. Returns {"loss": mean loss, "metric": mean metric or None}."""
    model.train()
    total_loss, total_metric, n = 0.0, 0.0, 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = model(x)
        loss = loss_fn(out, y)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss ({loss.item()}) at step {step_offset + i}: see Section 8")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        bs = y.shape[0]
        total_loss += loss.item() * bs
        n += bs
        if metric_fn is not None:
            total_metric += metric_fn(out.detach(), y)
        if writer is not None:
            writer.add_scalar("train/step_loss", loss.item(), step_offset + i)
            writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], step_offset + i)
    return {"loss": total_loss / n, "metric": total_metric / n if metric_fn is not None else None}


@torch.inference_mode()
def evaluate(model, loader, loss_fn=F.cross_entropy, metric_fn=accuracy_sum):
    """Mean loss and metric of `model` over `loader`, in eval mode and without autograd."""
    model.eval()
    total_loss, total_metric, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = model(x)
        total_loss += loss_fn(out, y).item() * y.shape[0]
        n += y.shape[0]
        if metric_fn is not None:
            total_metric += metric_fn(out, y)
    return {"loss": total_loss / n, "metric": total_metric / n if metric_fn is not None else None}


torch.manual_seed(SEED)
model = MLP(hidden_dims=(256, 128), dropout=0.2).to(device)
print("untrained model:", evaluate(model, val_loader))

An untrained model has a validation loss near $\ln 10 \approx 2.30$ and an accuracy near $0.1$, which is the first sanity check of Notebook 04's checklist.

### 3.2 `fit`: scheduling, clipping, early stopping, checkpointing, history

`fit` strings the two functions together over epochs and adds everything a real run needs: the scheduler (stepped per batch by default, or per epoch, or on the validation loss for `ReduceLROnPlateau`), gradient clipping, **early stopping** with a patience counter, saving the **best model** whenever the validation loss improves and restoring it at the end, per-epoch logging to TensorBoard (Section 4), a list of **callbacks** that receive the state after each epoch and can stop the run (Exercise 4), and a `history` dictionary with one list per quantity, which is what you plot afterwards. The `history` and `start_epoch` arguments let a run be continued from a checkpoint (Section 5), and `train_epoch_fn` lets you substitute a variant of the training step (mixup in Exercise 2) without touching the rest.

In [ ]:
def fit(model, train_loader, val_loader, optimizer, epochs, loss_fn=F.cross_entropy, scheduler=None,
        scheduler_mode="batch", clip_norm=None, patience=None, ckpt_path=None, writer=None,
        metric_fn=accuracy_sum, metric_name="acc", callbacks=(), train_epoch_fn=None,
        history=None, start_epoch=1, verbose=True):
    """Train for `epochs` epochs with validation after each one. Returns the history dict.

    scheduler_mode: "batch" (scheduler.step() after each batch), "epoch" (once per epoch) or
                    "plateau" (scheduler.step(val_loss), for ReduceLROnPlateau).
    patience:       stop when the validation loss has not improved for this many epochs (None = never).
    ckpt_path:      save model.state_dict() there whenever the validation loss improves, and reload
                    the best weights at the end.
    callbacks:      objects with on_epoch_end(state) -> bool; returning True stops training.
    history/start_epoch: pass the values from a checkpoint to continue an interrupted run.
    """
    train_epoch_fn = train_epoch_fn or train_one_epoch
    keys = ["train_loss", f"train_{metric_name}", "val_loss", f"val_{metric_name}", "lr", "epoch_time"]
    history = history if history is not None else {k: [] for k in keys}
    best_val = min(history["val_loss"], default=float("inf"))
    bad_epochs = 0
    steps_per_epoch = len(train_loader)
    for epoch in range(start_epoch, start_epoch + epochs):
        t0 = time.time()
        tr = train_epoch_fn(model, train_loader, optimizer, loss_fn=loss_fn,
                            scheduler=scheduler if scheduler_mode == "batch" else None,
                            clip_norm=clip_norm, metric_fn=metric_fn, writer=writer,
                            step_offset=(epoch - 1) * steps_per_epoch)
        va = evaluate(model, val_loader, loss_fn=loss_fn, metric_fn=metric_fn)
        if scheduler is not None and scheduler_mode == "epoch":
            scheduler.step()
        elif scheduler is not None and scheduler_mode == "plateau":
            scheduler.step(va["loss"])
        lr = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0
        for k, v in zip(keys, (tr["loss"], tr["metric"], va["loss"], va["metric"], lr, elapsed)):
            history[k].append(v)
        if writer is not None:
            writer.add_scalar("epoch/train_loss", tr["loss"], epoch)
            writer.add_scalar("epoch/val_loss", va["loss"], epoch)
            writer.add_scalar("epoch/lr", lr, epoch)
            if metric_fn is not None:
                writer.add_scalar(f"epoch/train_{metric_name}", tr["metric"], epoch)
                writer.add_scalar(f"epoch/val_{metric_name}", va["metric"], epoch)
        improved = va["loss"] < best_val - 1e-6
        if improved:
            best_val, bad_epochs = va["loss"], 0
            if ckpt_path is not None:
                torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1
        if verbose:
            metric_str = "" if metric_fn is None else f" | train {metric_name} {tr['metric']:.3f} | val {metric_name} {va['metric']:.3f}"
            print(f"epoch {epoch:3d} | train loss {tr['loss']:.4f} | val loss {va['loss']:.4f}{metric_str} | "
                  f"lr {lr:.2e} | {elapsed:5.1f}s{' *' if improved else ''}")
        state = dict(epoch=epoch, history=history, model=model, optimizer=optimizer, improved=improved)
        stop = any([bool(cb.on_epoch_end(state)) for cb in callbacks])
        if patience is not None and bad_epochs >= patience:
            if verbose:
                print(f"early stopping: no validation-loss improvement for {patience} epochs")
            stop = True
        if stop:
            break
    if ckpt_path is not None and Path(ckpt_path).exists():
        model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))     # best weights
    return history


def plot_history(history, metric_name="acc", title=""):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    ax1.plot(epochs, history["train_loss"], "o-", label="train"); ax1.plot(epochs, history["val_loss"], "o-", label="validation")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title(f"{title} loss".strip()); ax1.legend()
    ax2.plot(epochs, history[f"train_{metric_name}"], "o-", label="train"); ax2.plot(epochs, history[f"val_{metric_name}"], "o-", label="validation")
    ax2.set_xlabel("epoch"); ax2.set_ylabel(metric_name); ax2.set_title(f"{title} {metric_name}".strip()); ax2.legend()
    plt.tight_layout(); plt.show()


EPOCHS_DEMO = 1 if FAST_RUN else 3
torch.manual_seed(SEED)
model = MLP(hidden_dims=(256, 128), dropout=0.2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS_DEMO * len(train_loader))
history = fit(model, train_loader, val_loader, opt, EPOCHS_DEMO, scheduler=sched, clip_norm=1.0,
              patience=3, ckpt_path=CKPT_DIR / "mlp_demo_best.pt")
plot_history(history, title="MLP demo:")

One line per epoch, a star marking an improvement of the validation loss, and the history ready for plotting: this is the shape every training run in the rest of the course takes. Under `FAST_RUN` the demo is a single epoch on 6,000 images and reaches roughly 80% validation accuracy; the full run on 55,000 images reaches the high eighties within three epochs.

### 3.3 Metrics: accuracy, per-class accuracy, confusion matrix, classification report

Accuracy is a single number and hides a lot. Two cheap additions tell you *where* a classifier fails: the **per-class accuracy** (which classes are hard?) and the **confusion matrix** $C_{ij}$, the number of examples of true class $i$ predicted as class $j$, whose off-diagonal entries show which classes are confused with which. `scikit-learn` computes the confusion matrix and a **classification report** with precision, recall and F1 per class; per-class accuracy is a one-liner with `torch.bincount`. The helper `predict_all` collects predictions, targets and probabilities for a whole loader on the CPU, which every analysis below needs.

In [ ]:
@torch.inference_mode()
def predict_all(model, loader):
    """Predicted classes, true classes and softmax probabilities for every example in `loader` (on CPU)."""
    model.eval()
    preds, targets, probs = [], [], []
    for x, y in loader:
        out = model(x.to(device, non_blocking=True))
        probs.append(F.softmax(out, dim=1).cpu())
        preds.append(out.argmax(dim=1).cpu())
        targets.append(y)
    return torch.cat(preds), torch.cat(targets), torch.cat(probs)


def per_class_accuracy(preds, targets, n_classes):
    correct = torch.bincount(targets[preds == targets], minlength=n_classes)
    total = torch.bincount(targets, minlength=n_classes)
    return correct / total.clamp(min=1)


def plot_confusion(cm, classes, title="Confusion matrix"):
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar(fraction=0.046)
    plt.xticks(range(len(classes)), classes, rotation=60, ha="right", fontsize=8)
    plt.yticks(range(len(classes)), classes, fontsize=8)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title(title); plt.grid(False)
    plt.tight_layout(); plt.show()


preds, targets, probs = predict_all(model, val_loader)
pca = per_class_accuracy(preds, targets, len(CLASSES))
print("per-class validation accuracy:")
for c, a in zip(CLASSES, pca.tolist()):
    print(f"   {c:12s} {a:.3f}")
cm = confusion_matrix(targets.cpu().numpy(), preds.cpu().numpy())
plot_confusion(cm, CLASSES, title="MLP demo: validation confusion matrix")
print(classification_report(targets.cpu().numpy(), preds.cpu().numpy(), target_names=CLASSES, digits=3))

The confusion matrix has a clear structure: the hard classes are *Shirt*, *T-shirt/top*, *Pullover* and *Coat*, which are confused with each other, and the easy ones are *Trouser*, *Bag* and *Sandal/Sneaker/Ankle boot* (the footwear classes are confused only among themselves). This pattern is stable across models and is worth knowing: a "93% FashionMNIST model" is essentially a model that has become slightly better at telling shirts from T-shirts. The classification report adds **precision** (of the examples predicted as class $k$, how many are really $k$) and **recall** (of the true $k$'s, how many did we find); they diverge for imbalanced classes, which is why accuracy alone can be misleading.

## 4. Logging and experiment tracking with TensorBoard

### 4.1 `SummaryWriter`

Printing one line per epoch is fine for a notebook, but you will run dozens of experiments, and you want to compare them, look at curves during training, and keep the results. **TensorBoard** is the standard tool: a `SummaryWriter` writes event files to a directory, and the TensorBoard web app reads every run under a root folder and draws them together. A writer can log **scalars** (`add_scalar`: losses, metrics, the learning rate, gradient norms), **images** (`add_image`/`add_images`: a batch of inputs, reconstructions, attention maps), **histograms** (`add_histogram`: weights and gradients per layer, to catch vanishing or exploding scales), **text**, the **model graph** (`add_graph`), and **hyperparameters with their final metrics** (`add_hparams`, which makes the runs sortable in a table). Our `train_one_epoch` and `fit` already log to a writer when one is passed.

The run directory is worth some discipline: one folder per run, named with a timestamp and a short tag, containing the event files, the configuration (as JSON and, if PyYAML is available, YAML) and the checkpoints. The configuration is a plain dictionary; saving it next to the results is what makes a result reproducible three months later, when you no longer remember which learning rate produced it.

In [ ]:
def make_run_dir(tag, config):
    """Create RUNS_DIR/<timestamp>_<tag>/ and save the config there as JSON (and YAML when available)."""
    run_dir = RUNS_DIR / f"{datetime.now().strftime('%Y%m%d-%H%M%S')}_{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    if HAS_YAML:
        with open(run_dir / "config.yaml", "w") as f:
            yaml.safe_dump(config, f)
    return run_dir


config = dict(model="mlp", hidden_dims=[256, 128], dropout=0.2, lr=2e-3, weight_decay=0.01,
              batch_size=BATCH_SIZE, epochs=EPOCHS_DEMO, seed=SEED, n_train=N_TRAIN)
run_dir = make_run_dir("mlp_tb_demo", config)
writer = SummaryWriter(str(run_dir))

torch.manual_seed(config["seed"])
model = MLP(hidden_dims=tuple(config["hidden_dims"]), dropout=config["dropout"]).to(device)
xb, yb = next(iter(train_loader))
writer.add_images("inputs/first_batch", (xb[:32] * FM_STD + FM_MEAN).clamp(0, 1), global_step=0)
try:
    writer.add_graph(model, xb.to(device))
except Exception as e:                                       # graph tracing is fragile for some models
    print("add_graph failed (this is harmless):", str(e)[:80])
opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
history = fit(model, train_loader, val_loader, opt, config["epochs"], writer=writer, ckpt_path=run_dir / "best.pt")
for name, p in model.named_parameters():
    writer.add_histogram(f"weights/{name}", p.detach().cpu(), global_step=config["epochs"])
writer.add_text("notes", "Section 4 demo run: MLP with AdamW, no scheduler.")
writer.flush(); writer.close()
print("run directory:", run_dir)
print("contents:", sorted(p.name[:40] for p in run_dir.iterdir()))
print("config.json:", json.dumps(config))

To look at the run, start TensorBoard from a terminal in the `notebooks` folder (do not run it inside a notebook cell, it is a long-lived server):

```
tensorboard --logdir ../data/runs
```

and open the printed URL (usually http://localhost:6006). The *Scalars* tab shows `train/step_loss` and `train/lr` per step and the `epoch/*` curves per epoch for every run under `runs/`, with a run selector on the left; *Images* shows the input batch; *Histograms* shows the weight distributions; *Graphs* the traced model; *HParams* the table of configurations once you log them with `add_hparams` (Section 7 does). A few habits pay off: log the learning rate always (it catches scheduler bugs), log the gradient norm if you clip, log a few images with their predictions per epoch for vision tasks, and use a consistent naming scheme (`split/quantity`) so that curves from different runs line up.

> **Going deeper.** TensorBoard is local and simple. For teams and long projects, hosted trackers (Weights & Biases, MLflow, Neptune, Comet) add run comparison across machines, artifact storage and dashboards; they all consume the same `config` dictionary and the same per-epoch scalars, so nothing in `fit` needs to change beyond a different writer object.

## 5. Checkpointing and resuming

### 5.1 What a checkpoint must contain

`fit` saved the best *weights* (`model.state_dict()`), which is what you need for inference. To **resume** an interrupted run, exactly as if it had never stopped, you need more: the **optimizer state** (Adam's moment estimates, momentum buffers), the **scheduler state** (where in the schedule we are), the **epoch** counter and the history so far, the **configuration**, and the **random number generator states** of Python, NumPy, PyTorch (CPU and CUDA) and of the `DataLoader`'s shuffling generator, so that dropout masks, augmentations and the mini-batch order continue the same sequence. Without the optimizer state, a resumed Adam run starts with fresh, zero moments and its first steps are wrong; without the RNG states the run is still valid but not reproducible.

The functions below save and load all of that in one dictionary with `torch.save`/`torch.load`. Two options of `torch.load` matter. `map_location="cpu"` loads every tensor onto the CPU regardless of where it was saved, which is what makes a checkpoint from a GPU machine loadable on a laptop; `model.load_state_dict` then copies the values into the model's existing (device) tensors, and `optimizer.load_state_dict` moves its state to the parameters' device automatically. `weights_only=False` is needed because our checkpoint contains an object outside the allow-list of PyTorch's restricted unpickler: the NumPy generator state is a tuple that holds a NumPy array. Since PyTorch 2.6 the default is `weights_only=True`, which accepts tensors, plain Python containers of numbers and strings (our `history` dictionary would pass) and a short list of PyTorch types, and rejects everything else as a protection against malicious pickles, so **only load checkpoints you trust** with `weights_only=False`.

In [ ]:
def get_rng_states():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_states(states):
    random.setstate(states["python"]); np.random.set_state(states["numpy"]); torch.set_rng_state(states["torch"])
    if states["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(states["cuda"])


def save_checkpoint(path, model, optimizer, epoch, history, config=None, scheduler=None, loader_generator=None):
    """Everything needed to resume: weights, optimizer, scheduler, epoch, history, config and RNG states."""
    ckpt = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "history": history,
        "config": config,
        "rng": get_rng_states(),
        "loader_generator": loader_generator.get_state() if loader_generator is not None else None,
        "torch_version": str(torch.__version__),   # plain str: TorchVersion objects are not weights_only-safe
    }
    torch.save(ckpt, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, loader_generator=None, restore_rng=True):
    """Load a checkpoint written by save_checkpoint into existing objects; returns the checkpoint dict."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and ckpt["scheduler"] is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    if loader_generator is not None and ckpt["loader_generator"] is not None:
        loader_generator.set_state(ckpt["loader_generator"])
    if restore_rng:
        set_rng_states(ckpt["rng"])
    return ckpt


print("checkpoint helpers defined")

### 5.2 Experiment: resuming reproduces the uninterrupted run

The test of a checkpointing scheme is simple: run A trains for two epochs without interruption; run B trains for one epoch, saves a checkpoint, and is then continued in *fresh* objects (a newly constructed model and optimizer, created with a different seed on purpose, so that every piece of state must come from the checkpoint). If the checkpoint is complete, the second epoch of B is identical to the second epoch of A: same mini-batch order, same dropout masks, same weights, same losses. We use a 6,000-image subset and an MLP with dropout (which makes the test stricter, since dropout consumes random numbers).

In [ ]:
demo_set = Subset(train_set, list(range(min(len(train_set), DEMO_N))))


def new_training_objects(seed):
    torch.manual_seed(seed)
    model = MLP(hidden_dims=(256,), dropout=0.2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(seed)                     # drives the shuffling of this loader
    loader = DataLoader(demo_set, batch_size=BATCH_SIZE, shuffle=True, generator=gen, num_workers=0, pin_memory=PIN)
    return model, opt, gen, loader


# run A: two epochs, uninterrupted
model_a, opt_a, gen_a, loader_a = new_training_objects(SEED)
hist_a = fit(model_a, loader_a, val_loader, opt_a, epochs=2, verbose=False)

# run B: one epoch, checkpoint, then resume in fresh objects built with a different seed
model_b, opt_b, gen_b, loader_b = new_training_objects(SEED)
hist_b = fit(model_b, loader_b, val_loader, opt_b, epochs=1, verbose=False)
save_checkpoint(CKPT_DIR / "resume_demo.pt", model_b, opt_b, epoch=1, history=hist_b, loader_generator=gen_b)
del model_b, opt_b

model_c, opt_c, gen_c, loader_c = new_training_objects(seed=999)
ckpt = load_checkpoint(CKPT_DIR / "resume_demo.pt", model_c, opt_c, loader_generator=gen_c)
hist_c = fit(model_c, loader_c, val_loader, opt_c, epochs=1, history=ckpt["history"], start_epoch=ckpt["epoch"] + 1, verbose=False)

print(f"run A train loss per epoch: {[round(v, 6) for v in hist_a['train_loss']]}")
print(f"run B+resume:               {[round(v, 6) for v in hist_c['train_loss']]}")
print(f"max |difference| in train loss: {max(abs(a - b) for a, b in zip(hist_a['train_loss'], hist_c['train_loss'])):.2e}, "
      f"in val loss: {max(abs(a - b) for a, b in zip(hist_a['val_loss'], hist_c['val_loss'])):.2e}")
print(f"checkpoint file: {(CKPT_DIR / 'resume_demo.pt').stat().st_size / 1e6:.1f} MB "
      f"(model {sum(p.numel() for p in model_c.parameters()) * 4 / 1e6:.1f} MB + optimizer state, twice that for AdamW)")

The resumed run reproduces run A's second epoch exactly on the CPU (to the last digit; on a GPU, differences at the $10^{-6}$ level can appear from non-deterministic kernels, see Section 6). Note the size of the checkpoint: AdamW keeps two extra tensors per parameter, so the optimizer state is twice the size of the weights, and a full checkpoint is about three times the size of a weights-only one. This is why "best weights" and "resume" checkpoints are usually kept separately, and why Notebook 14 · Efficient Training and Scaling discusses optimizer memory.

### 5.3 Best-model selection, `state_dict` versus pickling, strict and partial loading

**Best model versus last model.** When `ckpt_path` is given, `fit` saves the weights at the epoch with the best validation loss and restores them at the end, so the returned model is the *best* one, not the *last* one; a resume checkpoint, in contrast, must be the last state. Keep both files, named accordingly.

**`state_dict` versus the whole model.** `torch.save(model, path)` pickles the entire object, including a reference to its class by *import path*. It works until you rename a class, move a file, or refactor, after which the checkpoint cannot be unpickled. `state_dict` checkpoints only contain tensors keyed by name, and can be loaded into any model with the same parameter names and shapes, which is what makes them robust and shareable. Always save state dicts.

**Strict and partial loading.** `load_state_dict` is strict by default: every key in the file must match a key in the model, with the same shape, or it raises. This catches accidental mismatches. When a mismatch is intended, for instance loading a pretrained body under a new classification head (transfer learning, Notebook 07 · Transfer Learning and Modern Vision), filter the dictionary and pass `strict=False`; the return value lists the `missing_keys` (in the model but not in the file, left at their initialization) and `unexpected_keys` (in the file but not in the model, ignored). Check both lists every time: an unexpected key is usually a naming mistake that silently leaves a layer untrained.

In [ ]:
# (1) map_location: the best weights of the Section 3 demo, loaded on the CPU and then into a device model
sd = torch.load(CKPT_DIR / "mlp_demo_best.pt", map_location="cpu")
print(f"state_dict loaded with map_location='cpu': {len(sd)} tensors, on {sd['head.weight'].device}")
restored = MLP(hidden_dims=(256, 128), dropout=0.2).to(device)
restored.load_state_dict(sd)
print(f"restored model on {next(restored.parameters()).device}: val acc {evaluate(restored, val_loader)['metric']:.3f}")

# (2) strict loading catches a mismatch: same body, a 3-class head
new_model = MLP(hidden_dims=(256, 128), dropout=0.2, n_classes=3).to(device)
try:
    new_model.load_state_dict(sd)
except RuntimeError as e:
    print("strict loading fails:", " ".join(str(e).split())[:140])

# (3) partial loading: keep the body, re-initialize the head
body_only = {k: v for k, v in sd.items() if not k.startswith("head.")}
result = new_model.load_state_dict(body_only, strict=False)
print("partial loading: missing keys", result.missing_keys, "| unexpected keys", result.unexpected_keys)

## 6. Reproducibility

### 6.1 Seeding everything

A deep learning run draws random numbers from at least four generators: Python's `random` (used by some data code), NumPy (augmentation libraries, our synthetic data), PyTorch's CPU generator (weight initialization, dropout, `randperm` for shuffling) and PyTorch's CUDA generators (dropout and sampling on the GPU). `torch.manual_seed` seeds the CPU generator and all CUDA devices; the others need their own calls. The function below seeds them all and, optionally, asks PyTorch for **deterministic algorithms**:

- `torch.backends.cudnn.deterministic = True` makes cuDNN choose deterministic convolution algorithms, and `torch.backends.cudnn.benchmark = False` stops it from auto-tuning (which can pick different algorithms on different runs).
- `torch.use_deterministic_algorithms(True)` makes every operation either use a deterministic implementation or raise an error (`warn_only=True` turns the error into a warning). Some CUDA operations have no deterministic implementation (for example the backward passes of `AdaptiveAvgPool2d`, `MaxPool3d` and bilinear `interpolate`, and `torch.histc`), and cuBLAS needs the environment variable `CUBLAS_WORKSPACE_CONFIG=:4096:8` set before the first CUDA call, which our setup cell does.

Two further sources of randomness live in the data pipeline: the `DataLoader`'s shuffling generator (pass `generator=torch.Generator().manual_seed(seed)` as we do) and, with `num_workers > 0`, the worker processes, which must be seeded with a `worker_init_fn` because each worker starts with its own state (the function is shown below; with `num_workers=0` it is never called).

In [ ]:
def seed_everything(seed, deterministic=False):
    """Seed Python, NumPy and PyTorch (CPU and CUDA); optionally request deterministic algorithms."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)                       # also seeds every CUDA device
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = False        # PyTorch's default; autotuning can pick different kernels per run
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def seed_worker(worker_id):
    """worker_init_fn for DataLoader(num_workers > 0): derive per-worker seeds from the torch seed."""
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def short_run(seed, deterministic):
    seed_everything(seed, deterministic=deterministic)
    model = MLP(hidden_dims=(128,), dropout=0.2).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    loader = DataLoader(demo_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN,
                        generator=torch.Generator().manual_seed(seed), worker_init_fn=seed_worker)
    h = fit(model, loader, val_loader, opt, epochs=1, verbose=False)
    return h["train_loss"][0], h["val_acc"][0]


run1 = short_run(SEED, deterministic=True)
run2 = short_run(SEED, deterministic=True)
run3 = short_run(SEED + 1, deterministic=True)
print(f"seed {SEED}, run 1: train loss {run1[0]:.6f}, val acc {run1[1]:.4f}")
print(f"seed {SEED}, run 2: train loss {run2[0]:.6f}, val acc {run2[1]:.4f}  -> identical: {run1 == run2}")
print(f"seed {SEED + 1}:    train loss {run3[0]:.6f}, val acc {run3[1]:.4f}  -> different seed, different run")
seed_everything(SEED, deterministic=False)        # back to the (faster) non-deterministic kernels for the rest of the notebook

Two runs with the same seed are identical to the last digit; a different seed gives a different run. The difference between the seeds is itself informative: it is the *noise floor* of your experiment, and any improvement smaller than the seed-to-seed variation is not an improvement. Serious comparisons therefore report the mean and standard deviation over several seeds (three to five is common), which is more valuable than bit-exact determinism.

### 6.2 What can still differ

Even with everything seeded, results can differ between machines and versions, and it is worth knowing why rather than being surprised:

- **Floating-point non-associativity.** Parallel reductions sum in a hardware-dependent order, so a GPU and a CPU, or two GPU models, give results that differ in the last bits; over thousands of steps those differences grow into visibly different runs. Determinism holds only for the same hardware, the same library versions and the same batch size.
- **Non-deterministic kernels.** Some CUDA operations use atomic additions whose order varies from run to run; `torch.use_deterministic_algorithms(True)` either replaces them with slower deterministic versions or raises so that you know which operation is responsible.
- **cuDNN autotuning** (`benchmark=True`) may select different convolution algorithms across runs.
- **Data pipeline order** with multiple workers depends on worker scheduling unless the sampler is seeded and the workers are initialized deterministically.
- **Library versions.** A different PyTorch, CUDA or cuDNN version can change default algorithms. Record them with the run (our checkpoint stores `torch.__version__`; `torch.utils.collect_env` prints the full environment).

The pragmatic policy: seed everything so that your own reruns match; enable deterministic mode when you are hunting a bug that depends on the exact numbers; do not pay its speed cost in routine training; and always evaluate over several seeds before claiming a difference.

## 7. A complete project: FashionMNIST

We now assemble everything into one function, `run_experiment`, that takes a configuration dictionary and does the whole job: seeds, creates the run directory with the config, builds the model and the data loaders, trains with AdamW, a one-cycle schedule and gradient clipping, checkpoints the best model, logs to TensorBoard, evaluates on the test set once, records the hyperparameters and final metrics with `add_hparams`, and writes a `results.json`. This is the shape of the `train.py` script of a real project (Section 9), and we use it for two models, an MLP and the small CNN, and then for a mini hyperparameter sweep.

Budgets: under `FAST_RUN` each model trains for one epoch on 6,000 images (a minute in total on a CPU); the full run trains both for 8 epochs on 55,000 images, where the MLP reaches about 89–90% test accuracy and the CNN about 92–93%.

In [ ]:
def build_model(config):
    if config["model"] == "mlp":
        return MLP(hidden_dims=tuple(config["hidden_dims"]), dropout=config["dropout"])
    if config["model"] == "cnn":
        return SmallCNN(dropout=config["dropout"])
    raise ValueError(f"unknown model type {config['model']!r}")


def run_experiment(config, verbose=True):
    """Train one configuration end to end and evaluate it once on the test set.

    Returns (model with the best validation weights, history, test metrics)."""
    seed_everything(config["seed"])
    run_dir = make_run_dir(config["tag"], config)
    writer = SummaryWriter(str(run_dir))
    model = build_model(config).to(device)
    loader = DataLoader(train_set if config["augment"] else train_set_plain, batch_size=config["batch_size"],
                        shuffle=True, num_workers=0, pin_memory=PIN,
                        generator=torch.Generator().manual_seed(config["seed"]))
    opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=config["lr"], total_steps=config["epochs"] * len(loader))
    t0 = time.time()
    history = fit(model, loader, val_loader, opt, config["epochs"], scheduler=sched, clip_norm=1.0,
                  ckpt_path=run_dir / "best.pt", writer=writer, verbose=verbose)
    test = evaluate(model, test_loader)
    hparams = {k: v for k, v in config.items() if isinstance(v, (int, float, str, bool))}
    writer.add_hparams(hparams, {"hparam/val_acc": max(history["val_acc"]), "hparam/test_acc": test["metric"]})
    writer.close()
    with open(run_dir / "results.json", "w") as f:
        json.dump({"history": history, "test": test, "train_time_s": time.time() - t0}, f, indent=2)
    if verbose:
        print(f"[{config['tag']}] best val acc {max(history['val_acc']):.4f} | test loss {test['loss']:.4f} | "
              f"test acc {test['metric']:.4f} | {time.time() - t0:.0f}s | run dir {run_dir.name}")
    return model, history, test


EPOCHS_MLP = 1 if FAST_RUN else 8
EPOCHS_CNN = 1 if FAST_RUN else 8
mlp_config = dict(tag="fmnist_mlp", model="mlp", hidden_dims=[512, 256], dropout=0.2, lr=2e-3, weight_decay=0.05,
                  batch_size=128, epochs=EPOCHS_MLP, augment=True, seed=SEED, n_train=N_TRAIN)
cnn_config = dict(tag="fmnist_cnn", model="cnn", hidden_dims=[], dropout=0.3, lr=3e-3, weight_decay=0.05,
                  batch_size=128, epochs=EPOCHS_CNN, augment=True, seed=SEED, n_train=N_TRAIN)

print("=== MLP ===")
mlp_model, mlp_hist, mlp_test = run_experiment(mlp_config)
print("=== CNN ===")
cnn_model, cnn_hist, cnn_test = run_experiment(cnn_config)

The CNN wins, and by a margin that grows with training time: convolutions exploit the two-dimensional structure of the image (locality, translation equivariance) that the MLP has to learn from scratch, with far fewer parameters per feature. Notebook 06 · Convolutional Neural Networks explains why. With only one epoch under `FAST_RUN` both models are far from converged, so treat those numbers as smoke-test values; the full run is where the 90% claim is made.

In [ ]:
plot_history(mlp_hist, title="MLP:")
plot_history(cnn_hist, title="CNN:")
print(f"test accuracy: MLP {mlp_test['metric']:.4f} | CNN {cnn_test['metric']:.4f}")

### 7.1 Error analysis: confusion matrix, per-class accuracy, confidently wrong examples

A single accuracy number is the beginning of the analysis, not the end. Three views of the CNN's test errors: the confusion matrix, the per-class accuracies of both models side by side, and a grid of the test images that the CNN gets wrong *with the highest confidence*, which is where labelling noise, genuinely ambiguous images and systematic model failures show up.

In [ ]:
preds, targets, probs = predict_all(cnn_model, test_loader)
cm = confusion_matrix(targets.cpu().numpy(), preds.cpu().numpy())
plot_confusion(cm, CLASSES, title="CNN: test confusion matrix")

preds_mlp, _, _ = predict_all(mlp_model, test_loader)
pca_cnn = per_class_accuracy(preds, targets, len(CLASSES))
pca_mlp = per_class_accuracy(preds_mlp, targets, len(CLASSES))
xpos = np.arange(len(CLASSES))
plt.figure(figsize=(9, 3.8))
plt.bar(xpos - 0.2, pca_mlp.cpu().numpy(), width=0.4, label="MLP")
plt.bar(xpos + 0.2, pca_cnn.cpu().numpy(), width=0.4, label="CNN")
plt.xticks(xpos, CLASSES, rotation=45, ha="right"); plt.ylabel("test accuracy"); plt.ylim(0.5, 1.0)
plt.title("Per-class test accuracy"); plt.legend(); plt.tight_layout(); plt.show()

The confusion matrix and the per-class chart both come from the CPU tensors that `predict_all` returned, and the same `preds`, `targets` and `probs` serve the third view: the misclassified test images, sorted by the probability the model assigned to its wrong prediction.

In [ ]:
wrong = (preds != targets).nonzero().squeeze(1)
confidence_in_wrong = probs[wrong, preds[wrong]]
show = wrong[confidence_in_wrong.argsort(descending=True)][:16]
fig, axes = plt.subplots(2, 8, figsize=(13, 4.2))
for ax, i in zip(axes.flat, show.tolist()):
    img, _ = test_set[i]
    ax.imshow((img[0] * FM_STD + FM_MEAN).cpu().numpy(), cmap="gray")
    ax.set_title(f"true: {CLASSES[int(targets[i])]}\npred: {CLASSES[int(preds[i])]} ({probs[i, preds[i]]:.2f})", fontsize=7)
    ax.axis("off")
plt.suptitle(f"CNN: the 16 most confident errors out of {len(wrong)} on the test set", fontsize=10)
plt.tight_layout(); plt.show()

The per-class chart makes the earlier observation quantitative: both models are near-perfect on trousers, bags and sandals, and the CNN's advantage is concentrated in the shirt/T-shirt/pullover/coat cluster. The confident errors are instructive too: many are images where the true label is arguable (a sleeveless "shirt" that looks like a T-shirt, a "coat" indistinguishable from a pullover), i.e. label noise and genuine ambiguity, which sets a ceiling on achievable accuracy that no model can pass. Looking at such grids is the single most useful habit in applied deep learning: it tells you whether to work on the model, the data, or the labels.

### 7.2 A mini hyperparameter sweep

Finally, the loop that `run_experiment` was built for: a small grid over the learning rate and the weight decay of the MLP, with short budgets, logged as separate runs so that TensorBoard's *HParams* tab can rank them. Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization argued for random search over grids; here the grid is tiny and serves to show the mechanics. Under `FAST_RUN` each configuration trains for one epoch; the full run uses three.

In [ ]:
SWEEP_EPOCHS = 1 if FAST_RUN else 3
sweep_results = []
print(f"{'lr':>8s} {'weight decay':>13s} {'best val acc':>13s} {'test acc':>9s}")
for lr in (5e-4, 2e-3, 8e-3):
    for wd in (0.0, 0.1):
        cfg = dict(mlp_config, tag=f"sweep_lr{lr:g}_wd{wd:g}", lr=lr, weight_decay=wd, epochs=SWEEP_EPOCHS, hidden_dims=[256, 128])
        _, h, t = run_experiment(cfg, verbose=False)
        sweep_results.append(dict(lr=lr, weight_decay=wd, val_acc=max(h["val_acc"]), test_acc=t["metric"]))
        print(f"{lr:8.0e} {wd:13.2f} {max(h['val_acc']):13.4f} {t['metric']:9.4f}")
best_cfg = max(sweep_results, key=lambda r: r["val_acc"])
print(f"\nbest by validation accuracy: lr {best_cfg['lr']:g}, weight decay {best_cfg['weight_decay']:g} "
      f"-> test acc {best_cfg['test_acc']:.4f}")

Two things to notice. The ranking must be made on the *validation* accuracy; the test column is printed only to show that the two agree, and in a real project you would evaluate the test set once, for the selected configuration. And with such short budgets the learning rate dominates (the largest one is often best for one epoch of one-cycle training), whereas weight decay only pays off over longer runs; ranking hyperparameters on short proxies systematically favours "fast starters", a bias that the Optuna pruners of Notebook 14 · Efficient Training and Scaling have to account for.

## 8. Debugging deep learning code

Deep learning bugs come in two kinds: those that crash (shape and device errors, exceptions in autograd) and those that do not (a model that trains but learns the wrong thing). The first kind is easy once you can read the error message; the second kind is found with sanity checks and instrumentation. This section covers both.

### 8.1 Shape errors and how to read them

The most common crash is a matrix-multiplication shape mismatch. Its message tells you the shapes involved; the skill is mapping them back to your code. In the example below a linear layer expecting 784 inputs receives an image batch of shape $(2, 1, 28, 28)$: `nn.Linear` acts on the *last* dimension, so it sees $2\cdot 1\cdot 28 = 56$ rows of $28$ features and complains that $28 \neq 784$. The fix is a `flatten`, which our `MLP` does internally. Three habits prevent most shape bugs: write the expected shape as a comment on every line of `forward` that changes it (`# (B, C, H, W) -> (B, C*H*W)`), `assert` the shape of the input at the top of `forward` with a message that prints the actual shape (as `SmallCNN` does), and run `torchinfo.summary` or a single dummy batch through a new model before training it.

In [ ]:
try:
    nn.Linear(784, 10).to(device)(torch.randn(2, 1, 28, 28, device=device))
except RuntimeError as e:
    print("RuntimeError:", str(e)[:100])

try:
    cnn_model(torch.randn(2, 784, device=device))          # the assert in SmallCNN.forward
except AssertionError as e:
    print("AssertionError:", e)

# a dummy forward pass is the cheapest test of a new architecture
with torch.inference_mode():
    out = cnn_model.eval()(torch.zeros(3, 1, 28, 28, device=device))
print("dummy batch through the CNN: output", tuple(out.shape), "finite:", torch.isfinite(out).all().item())

### 8.2 Anomaly detection

When a loss becomes `nan`, the question is *where* it first appeared. `torch.autograd.detect_anomaly()` makes autograd check the output of every backward function and raise, with a traceback pointing at the forward operation responsible, as soon as one produces a `nan` or `inf`. It slows training down several-fold, so it is a debugging tool, not a default. The example below is a classic: the forward pass is perfectly finite ($0 \cdot \sqrt{0} = 0$), but the gradient of $\sqrt{x}$ at $x = 0$ is infinite and $0 \cdot \infty$ is `nan`.

In [ ]:
x = torch.tensor([0.0, 1.0, 4.0], requires_grad=True)
mask = torch.tensor([0.0, 1.0, 1.0])
loss = (torch.sqrt(x) * mask).sum()          # forward: finite (0*0 + 1 + 2 = 3)
loss.backward()
print("forward value:", loss.item(), "| gradient:", x.grad.tolist(), "<- nan from 0 * inf, silently")

x.grad = None
try:
    with torch.autograd.detect_anomaly():
        loss = (torch.sqrt(x) * mask).sum()
        loss.backward()
except RuntimeError as e:
    print("with detect_anomaly:", str(e).splitlines()[0])

The message names the backward function (`SqrtBackward0`), and in a notebook the full traceback shows the line of the forward pass that created it. The usual fixes are an $\epsilon$ inside the square root, a `clamp`, or `torch.where` with a safe branch. The same mechanism catches `log(0)`, divisions by zero, and overflows.

### 8.3 Hooks: looking inside the network

**Hooks** are functions that PyTorch calls at specific moments: a **forward hook** on a module receives its inputs and output after every forward pass, a **full backward hook** receives the gradients with respect to its inputs and output during backpropagation, and a **tensor hook** receives the gradient of a specific tensor. They let you inspect activations and gradients without changing the model, and they are the basis of feature visualization and Grad-CAM (Notebook 07 · Transfer Learning and Modern Vision). A hook registration returns a handle whose `remove()` you must call afterwards, otherwise the hook keeps running (and keeps references alive) for the life of the model.

The cell below attaches a forward hook to every convolution, linear layer and ReLU of the CNN and records the shape, mean, standard deviation and fraction of zeros of each output on one batch; then it backpropagates a loss and prints the gradient norm of every parameter tensor. Together, these two tables are the "is anything vanishing, exploding or dead?" check of Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization, now for an arbitrary model.

In [ ]:
activation_stats = {}


def make_forward_hook(name):
    def hook(module, inputs, output):
        out = output.detach()
        activation_stats[name] = dict(shape=tuple(out.shape), mean=out.mean().item(), std=out.std().item(),
                                      zero_frac=(out == 0).float().mean().item())
    return hook


handles = [m.register_forward_hook(make_forward_hook(name)) for name, m in cnn_model.named_modules()
           if isinstance(m, (nn.Conv2d, nn.Linear, nn.ReLU))]
xb, yb = next(iter(val_loader))
xb, yb = xb[:64].to(device), yb[:64].to(device)
cnn_model.eval()
loss = F.cross_entropy(cnn_model(xb), yb)
loss.backward()
for h in handles:
    h.remove()                                                  # always clean up

print(f"{'module':22s} {'output shape':>18s} {'mean':>8s} {'std':>8s} {'zeros':>7s}")
for name, s in activation_stats.items():
    print(f"{name:22s} {str(s['shape']):>18s} {s['mean']:8.3f} {s['std']:8.3f} {s['zero_frac']:7.2f}")
print("\ngradient norm per parameter tensor:")
for name, p in cnn_model.named_parameters():
    print(f"   {name:24s} {tuple(p.shape)!s:>18s}  ||grad|| = {p.grad.norm().item():.2e}")
cnn_model.zero_grad(set_to_none=True)

# tensor hooks and backward hooks: a gradient passing through a specific tensor / module
z = torch.randn(3, requires_grad=True)
handle = z.register_hook(lambda g: print("tensor hook: gradient reaching z =", g.tolist()))
(z * torch.tensor([1.0, 2.0, 3.0])).sum().backward()
handle.remove()

The ReLU rows show the fraction of zero activations (a healthy network has a substantial but not overwhelming fraction, say 30–70%; close to 100% means dead units), and the gradient norms show whether the early layers still receive signal. A tensor hook is the lightest way to watch a single gradient; a module's `register_full_backward_hook` does the same for a whole layer. The pattern generalizes to *modifying* what flows through: a hook may return a new tensor, which is how gradient clipping per layer or feature ablations are implemented.

### 8.4 Overfitting a single batch with the real pipeline

Notebook 04 introduced the overfit-one-batch test on in-memory tensors; here it is with the real model, loader and `fit`, which is how you should run it on a new project: build a `DataLoader` over a single batch, use it as both training and validation set, and check that the loss goes to (almost) zero. If it does not, the bug is in the model, the loss, or the data, not in the hyperparameters. Note that BatchNorm and dropout are active during the training steps but not during the evaluation: once BatchNorm's running statistics have caught up with the batch (they are exponential moving averages with momentum $0.1$, so this takes a few dozen steps), the loss on the same batch in eval mode is even lower than the training loss, whereas in the first steps it can be higher because the running statistics still lag behind.

In [ ]:
ONE_BATCH_EPOCHS = 60 if FAST_RUN else 150
one_batch = TensorDataset(xb.cpu(), yb.cpu())
one_loader = DataLoader(one_batch, batch_size=64, shuffle=False)
seed_everything(SEED)
tiny = SmallCNN().to(device)
opt = torch.optim.Adam(tiny.parameters(), lr=1e-3)
h = fit(tiny, one_loader, one_loader, opt, ONE_BATCH_EPOCHS, verbose=False)
plt.figure(); plt.semilogy(h["train_loss"], label="train (one batch, BN/dropout on)"); plt.semilogy(h["val_loss"], label="same batch in eval mode")
plt.xlabel("step"); plt.ylabel("loss (log)"); plt.title("Overfit-one-batch test with the real pipeline"); plt.legend()
plt.tight_layout(); plt.show()
print(f"after {ONE_BATCH_EPOCHS} steps: loss on the batch {h['val_loss'][-1]:.4f}, accuracy {h['val_acc'][-1]:.3f}")

### 8.5 Common exceptions and what they mean

| Message (abbreviated) | Cause | Fix |
|---|---|---|
| `Expected all tensors to be on the same device` | model on GPU, data on CPU (or the reverse); a tensor attribute that is not a buffer | `.to(device)` on the data; `register_buffer` for constants |
| `mat1 and mat2 shapes cannot be multiplied (AxB and CxD)` | wrong input size for a linear layer; forgot to flatten | check `B == C`; add `flatten`; use `torchinfo.summary` |
| `CUDA out of memory` | batch, model or sequence too large; graphs kept alive by storing `loss` instead of `loss.item()` | smaller batch, `loss.item()`, `del` unused tensors, mixed precision (Notebook 14 · Efficient Training and Scaling) |
| `element 0 of tensors does not require grad` | `backward()` on a tensor with no graph: computed under `no_grad`, on detached data, or the parameters were frozen | check `requires_grad`; remove the `no_grad` |
| `Trying to backward through the graph a second time` | calling `backward()` twice on the same graph (e.g. a loss accumulated across steps) | recompute the loss per step, or `retain_graph=True` when really intended |
| `one of the variables needed for gradient computation has been modified by an inplace operation` | an in-place op (`x += ...`, `relu_`) on a tensor autograd still needs | replace by the out-of-place op |
| `Expected more than 1 value per channel when training` | BatchNorm in train mode with a batch of size 1 | `model.eval()` for inference; `drop_last=True` for training |
| `expected scalar type Long but found Float` (in `cross_entropy`) | class-index targets of shape `(B,)` have a floating-point dtype | `y.long()` (float targets with the same shape as the logits are instead interpreted as class probabilities and do not raise) |
| `RuntimeError: DataLoader worker ... exited unexpectedly` | worker processes failing (Windows notebooks, unpicklable dataset) | `num_workers=0` in notebooks; scripts with `if __name__ == "__main__":` |
| silent: loss stuck at `ln K` | learning rate zero or no `optimizer.step()`; parameters not registered (plain list); `zero_grad` missing | check `parameters()` count; overfit one batch |
| silent: eval metrics change between calls | `model.eval()` missing (dropout/BatchNorm active) | `model.eval()` and `inference_mode()` in every evaluation |

### 8.6 A debugging checklist

1. Run a dummy batch through the model and check the output shape and dtype.
2. Check the data: one batch, its shapes, dtypes, value range, and the labels (plot a few examples with their labels).
3. Compute the loss on the untrained model; it should be close to the value of a uniform prediction.
4. Overfit one batch. If it fails, stop and find the bug.
5. Count the parameters the optimizer sees and compare with the model's.
6. Confirm the modes: `train()` in the loop, `eval()` and `inference_mode()` in evaluation.
7. Log the learning rate and the gradient norm every step for the first run.
8. If the loss is `nan`: lower the learning rate first; then `detect_anomaly()`.
9. If training is fine but the metric is not: check the metric code on a hand-made example where you know the answer.
10. Reduce: smaller model, smaller data, `FAST_RUN`, one worker. Bugs are easier to see when everything is small.

## 9. Project structure for real work

A notebook is the right place to explore, and the wrong place to keep code you rely on: cells run out of order, state leaks between experiments, functions get redefined, and nothing can be imported or tested. The transition to a real project is mostly a reorganization of the pieces you already have in this notebook. A typical layout:

```
fashion-project/
  pyproject.toml            # package metadata and dependencies (or environment.yml)
  README.md
  configs/
    mlp.yaml                # one file per experiment family: model, data, optimizer, schedule, seed
    cnn.yaml
  src/fashion/              # an installable package (pip install -e .)
    __init__.py
    data.py                 # Dataset classes, transforms, build_loaders(config)
    models.py               # MLP, SmallCNN, build_model(config)
    engine.py               # train_one_epoch, evaluate, fit, callbacks
    checkpoint.py           # save_checkpoint, load_checkpoint, seed_everything
    metrics.py              # accuracy, per-class accuracy, confusion matrix helpers
    utils.py                # logging, run directories, config loading
  scripts/
    train.py                # parse args / load a config -> run_experiment
    evaluate.py             # load a checkpoint, evaluate on the test set, write a report
    sweep.py                # loop over configs (or call Optuna, Notebook 14)
  tests/
    test_models.py          # shapes and parameter counts for each model
    test_data.py            # a dataset returns the right types and shapes
    test_engine.py          # overfit-one-batch as an automated test
  notebooks/                # exploration and figures, importing from src/
  runs/                     # TensorBoard logs, configs and checkpoints (git-ignored)
```

The mapping from this notebook is direct: Section 1 becomes `models.py`, Section 2 `data.py`, Sections 3–4 `engine.py`, Sections 5–6 `checkpoint.py`, Section 7 `scripts/train.py`, and Section 8's checks become `tests/`. A few principles make the result pleasant to work with:

**Configuration drives everything.** A run is a config file plus a git commit. `train.py` loads the YAML (or JSON) into a dictionary, `build_model(config)` and `build_loaders(config)` construct the objects, and the config is copied into the run directory, as `make_run_dir` does. Command-line overrides (`--lr 3e-3`) are convenient for sweeps; libraries such as Hydra or plain `argparse` both work. A dataclass with typed fields catches typos in config keys, which are otherwise a classic silent bug.

```python
# scripts/train.py (sketch, not executed here)
import argparse, yaml
from fashion.data import build_loaders
from fashion.models import build_model
from fashion.engine import fit
from fashion.checkpoint import seed_everything

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--epochs", type=int)
    args = parser.parse_args()
    with open(args.config) as f:
        config = yaml.safe_load(f)
    if args.epochs is not None:
        config["epochs"] = args.epochs
    seed_everything(config["seed"])
    train_loader, val_loader, test_loader = build_loaders(config)
    model = build_model(config).to(config["device"])
    ...

if __name__ == "__main__":      # required for DataLoader workers on Windows
    main()
```

**The engine knows nothing about the task.** `fit` takes a model, loaders, an optimizer and a loss; it does not know whether it is training a classifier or a language model. That is what makes it reusable across the remaining notebooks and across projects.

**Tests are cheap and catch the expensive bugs.** A test that builds each model and pushes a dummy batch through it takes a second and catches every shape regression. A test that overfits one batch in fifty steps takes ten seconds and catches broken losses, frozen parameters and wrong labels. Run them with `pytest` before every long training job.

```python
# tests/test_models.py (sketch)
import torch
from fashion.models import MLP, SmallCNN

def test_output_shapes():
    x = torch.zeros(2, 1, 28, 28)
    for model in (MLP(), SmallCNN()):
        assert model(x).shape == (2, 10)
```

**Runs are artifacts.** Every run directory holds the config, the metrics, the checkpoints and the TensorBoard events; nothing about a result should live only in your memory or in a cell output. The `results.json` written by `run_experiment` is what a later notebook reads to produce the figures for a report, and it is what a colleague reads to reproduce your numbers.

**Notebooks remain, but for exploration.** They import from `src/`, they are short, and they are re-run top to bottom before anything in them is trusted. Notebook 17 · Capstone Projects builds a full `Trainer` class along these lines and uses it for the course projects.

## Exercises

**Exercise 1 (warm-up).** Write a `Dataset` for a synthetic *regression* table (three input features, one continuous target), split it into training and validation parts, and train a small MLP on it with the `fit` function: use `F.mse_loss` as the loss and the mean absolute error as the metric (`metric_fn` must return the *sum* over the batch, `metric_name="mae"`). Put the input standardization inside the model with the `Standardize` module of Section 1.2, so that the saved `state_dict` carries the preprocessing statistics. Confirm that the statistics appear in the `state_dict`.

**Solution to Exercise 1**

The dataset generates its data in `__init__` and returns the target with shape `(1,)` so that batches have shape `(B, 1)`, matching the model output for `F.mse_loss`. The standardization statistics are computed on the training split only and handed to the `Standardize` module; because they are buffers, they are saved and loaded with the weights and moved to the device with the model, and inference code never has to remember them.

In [ ]:
class RegressionTable(Dataset):
    """y = sin(x0) + 0.5 x1^2 - x2 + noise, with inputs on very different scales."""
    def __init__(self, n=2000, seed=SEED):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n, 3, generator=g) * torch.tensor([3.0, 1.0, 0.5]) + torch.tensor([0.0, 2.0, -1.0])
        noise = 0.1 * torch.randn(n, generator=g)
        self.y = (torch.sin(self.X[:, 0]) + 0.5 * self.X[:, 1] ** 2 - self.X[:, 2] + noise).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def mae_sum(outputs, targets):
    return (outputs - targets).abs().sum().item()


reg = RegressionTable()
train_reg, val_reg = random_split(reg, [1600, 400], generator=torch.Generator().manual_seed(SEED))
X_tr = reg.X[train_reg.indices]
torch.manual_seed(SEED)
model_reg = nn.Sequential(Standardize(X_tr.mean(dim=0), X_tr.std(dim=0)),
                          nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)
opt = torch.optim.Adam(model_reg.parameters(), lr=3e-3)
REG_EPOCHS = 10 if FAST_RUN else 40
h_reg = fit(model_reg, DataLoader(train_reg, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEED)),
            DataLoader(val_reg, batch_size=400), opt, REG_EPOCHS, loss_fn=F.mse_loss,
            metric_fn=mae_sum, metric_name="mae", verbose=False)
print(f"validation MSE {h_reg['val_loss'][-1]:.4f}, MAE {h_reg['val_mae'][-1]:.4f} (target std {reg.y.std().item():.3f})")
print("state_dict keys:", list(model_reg.state_dict().keys())[:4], "...")
plot_history(h_reg, metric_name="mae", title="Regression:")

The MAE ends far below the target's standard deviation, so the network has learned the non-linear relation, and the first two `state_dict` entries are the standardization mean and std. Without the standardization the feature with scale $3$ would dominate the early training and the one with scale $0.5$ would be nearly ignored; try removing the `Standardize` layer and compare.

**Exercise 2.** Implement **mixup** (Zhang et al., 2018) as an alternative `train_one_epoch`: for each batch draw $\lambda \sim \text{Beta}(\alpha, \alpha)$, form $\tilde x = \lambda x + (1-\lambda)\, x_{\pi}$ with a random permutation $\pi$ of the batch, and use the loss $\lambda\, \ell(f(\tilde x), y) + (1-\lambda)\, \ell(f(\tilde x), y_{\pi})$. Plug it into `fit` through the `train_epoch_fn` argument and compare an MLP trained with and without mixup on the same budget.

**Solution to Exercise 2**

Mixup trains on convex combinations of pairs of examples and their labels, which encourages linear behaviour between training examples and is a strong regularizer for small datasets. The function below has the same signature as `train_one_epoch` (plus `alpha`), so `fit` can use it unchanged; the training accuracy is measured against the dominant label of each mixed pair, which is the usual convention and is only a rough indicator.

In [ ]:
def train_one_epoch_mixup(model, loader, optimizer, loss_fn=F.cross_entropy, scheduler=None, clip_norm=None,
                          metric_fn=accuracy_sum, writer=None, step_offset=0, alpha=0.4):
    """train_one_epoch with mixup: convex combinations of pairs of examples and of their losses."""
    model.train()
    total_loss, total_metric, n = 0.0, 0.0, 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        lam = float(np.random.beta(alpha, alpha))
        perm = torch.randperm(x.shape[0], device=device)
        x_mix = lam * x + (1 - lam) * x[perm]
        out = model(x_mix)
        loss = lam * loss_fn(out, y) + (1 - lam) * loss_fn(out, y[perm])
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at step {step_offset + i}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        bs = y.shape[0]
        total_loss += loss.item() * bs
        n += bs
        if metric_fn is not None:
            total_metric += metric_fn(out.detach(), y if lam >= 0.5 else y[perm])
        if writer is not None:
            writer.add_scalar("train/step_loss", loss.item(), step_offset + i)
    return {"loss": total_loss / n, "metric": total_metric / n if metric_fn is not None else None}


MIX_EPOCHS = 1 if FAST_RUN else 5
for name, epoch_fn in [("plain", None), ("mixup (alpha=0.4)", train_one_epoch_mixup)]:
    seed_everything(SEED)
    model = MLP(hidden_dims=(512, 256), dropout=0.0).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
    loader = DataLoader(train_set_plain, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN,
                        generator=torch.Generator().manual_seed(SEED))
    h = fit(model, loader, val_loader, opt, MIX_EPOCHS, train_epoch_fn=epoch_fn, verbose=False)
    print(f"{name:18s}: final train acc {h['train_acc'][-1]:.3f} | val loss {h['val_loss'][-1]:.4f} | val acc {h['val_acc'][-1]:.3f}")

Mixup lowers the training accuracy (the network is asked to predict blends) and typically gives a lower validation *loss* and better calibration; the validation accuracy gain appears with longer training and is small on this easy task. Its main use is as a regularizer for small datasets and in fine-tuning (Notebook 07 · Transfer Learning and Modern Vision revisits mixup and CutMix for fine-tuning).

**Exercise 3.** Implement a `PeriodicCheckpoint` callback for `fit` that writes a full resume checkpoint (Section 5) at the end of every epoch, and a function `resume_from_checkpoint(path, model, optimizer, ..., total_epochs)` that restores everything and trains the remaining epochs. Verify, as in Section 5.2, that a run interrupted after two of three epochs and resumed in fresh objects (including a fresh scheduler) reproduces the uninterrupted three-epoch run, including the learning-rate trajectory.

**Solution to Exercise 3**

The callback receives the `state` dictionary from `fit` (epoch, history, model, optimizer) and needs references to the scheduler and the loader's generator, which it takes in its constructor. `resume_from_checkpoint` loads the checkpoint into freshly built objects, reads the epoch counter, and calls `fit` with `history` and `start_epoch` set, for the number of epochs that remain. The scheduler's state (its step counter) is part of the checkpoint, so the cosine schedule continues where it stopped.

In [ ]:
class PeriodicCheckpoint:
    """Callback: write a full resume checkpoint every `every` epochs."""
    def __init__(self, path, optimizer, scheduler=None, loader_generator=None, config=None, every=1):
        self.path, self.optimizer, self.scheduler = Path(path), optimizer, scheduler
        self.loader_generator, self.config, self.every = loader_generator, config, every

    def on_epoch_end(self, state):
        if state["epoch"] % self.every == 0:
            save_checkpoint(self.path, state["model"], self.optimizer, state["epoch"], state["history"],
                            config=self.config, scheduler=self.scheduler, loader_generator=self.loader_generator)
        return False                                   # never asks to stop


def resume_from_checkpoint(path, model, optimizer, train_loader, val_loader, total_epochs,
                           scheduler=None, loader_generator=None, **fit_kwargs):
    """Restore model/optimizer/scheduler/RNG/loader state from `path` and train the remaining epochs."""
    ckpt = load_checkpoint(path, model, optimizer, scheduler=scheduler, loader_generator=loader_generator)
    done, history = ckpt["epoch"], ckpt["history"]
    print(f"resumed after epoch {done}; {max(0, total_epochs - done)} epochs remaining")
    if total_epochs <= done:
        return history
    return fit(model, train_loader, val_loader, optimizer, total_epochs - done, scheduler=scheduler,
               history=history, start_epoch=done + 1, **fit_kwargs)


TOTAL = 3


def build_for_resume(seed):
    torch.manual_seed(seed)
    model = MLP(hidden_dims=(256,), dropout=0.2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
    gen = torch.Generator().manual_seed(seed)
    loader = DataLoader(demo_set, batch_size=BATCH_SIZE, shuffle=True, generator=gen, num_workers=0, pin_memory=PIN)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TOTAL * len(loader))
    return model, opt, gen, loader, sched


# uninterrupted reference run
m, o, g, ld, s = build_for_resume(SEED)
hist_ref = fit(m, ld, val_loader, o, TOTAL, scheduler=s, verbose=False)

# interrupted run: two epochs with periodic checkpoints, then "crash"
m, o, g, ld, s = build_for_resume(SEED)
ckpt_path = CKPT_DIR / "exercise3_resume.pt"
hist_partial = fit(m, ld, val_loader, o, 2, scheduler=s, verbose=False,
                   callbacks=[PeriodicCheckpoint(ckpt_path, o, scheduler=s, loader_generator=g)])
del m, o, g, ld, s

# resume in fresh objects built with an unrelated seed
m2, o2, g2, ld2, s2 = build_for_resume(seed=7)
hist_resumed = resume_from_checkpoint(ckpt_path, m2, o2, ld2, val_loader, TOTAL, scheduler=s2, loader_generator=g2, verbose=False)
for key in ("train_loss", "val_loss", "lr"):
    gap = max(abs(a - b) for a, b in zip(hist_ref[key], hist_resumed[key]))
    print(f"{key:10s} reference {[round(v, 5) for v in hist_ref[key]]} | resumed {[round(v, 5) for v in hist_resumed[key]]} | max gap {gap:.1e}")

All three epochs match, including the learning rate at the end of each epoch, which shows that the scheduler's position in the cosine cycle was restored together with the weights and the optimizer moments. In a real training script the callback is the only addition needed to make a multi-day run survive a pre-emption; the script starts with "if a checkpoint exists in the run directory, resume from it".

**Exercise 4.** Add callback support of your own to the loop: implement a `TimeLimit(seconds)` callback that stops training once the wall-clock budget is exhausted, a `StopAtMetric(name, target)` callback that stops when a validation metric reaches a target, and a `LambdaCallback(fn)` wrapper for one-off logic. Demonstrate them together on a short run.

**Solution to Exercise 4**

`fit` calls `on_epoch_end(state)` on every callback after each epoch and stops if any returns `True`; that protocol is all the three classes need. The demonstration uses a time limit of zero seconds so that it triggers deterministically after the first epoch, and a metric target that is unlikely to be met in one epoch, so that the printed reason is the time limit.

In [ ]:
class TimeLimit:
    """Stop training when more than `seconds` have elapsed since the callback was created."""
    def __init__(self, seconds):
        self.seconds, self.t0 = seconds, time.time()

    def on_epoch_end(self, state):
        elapsed = time.time() - self.t0
        if elapsed > self.seconds:
            print(f"TimeLimit: {elapsed:.1f}s > {self.seconds}s budget, stopping after epoch {state['epoch']}")
            return True
        return False


class StopAtMetric:
    """Stop training when history[name][-1] reaches `target`."""
    def __init__(self, name, target):
        self.name, self.target = name, target

    def on_epoch_end(self, state):
        value = state["history"][self.name][-1]
        if value >= self.target:
            print(f"StopAtMetric: {self.name} = {value:.3f} >= {self.target}, stopping after epoch {state['epoch']}")
            return True
        return False


class LambdaCallback:
    def __init__(self, fn):
        self.fn = fn

    def on_epoch_end(self, state):
        return bool(self.fn(state))


seed_everything(SEED)
model = MLP(hidden_dims=(256,), dropout=0.2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
loader = DataLoader(demo_set, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED),
                    num_workers=0, pin_memory=PIN)
report = LambdaCallback(lambda st: print(f"   callback: best val acc so far {max(st['history']['val_acc']):.3f}"))
h = fit(model, loader, val_loader, opt, epochs=5, verbose=True,
        callbacks=[report, StopAtMetric("val_acc", 0.99), TimeLimit(seconds=0.0)])
print(f"trained for {len(h['train_loss'])} of 5 requested epochs")

The run stops after the first epoch because of the time limit, after the reporting callback has printed. This is the whole design of callback systems such as those in PyTorch Lightning or Keras, reduced to its essence: a training loop with fixed hook points, and small objects that react to the state at those points without the loop knowing anything about them.

**Exercise 5 (challenge).** Implement **k-fold cross-validation** on a FashionMNIST subset with the pipeline of this notebook: split a subset of the training indices into $k$ folds, train $k$ models (each with a different fold as validation set and augmentation on the training folds only), and report the mean and standard deviation of the validation accuracy. Use it to compare two configurations (an MLP with and without dropout) and decide whether the difference is larger than the fold-to-fold spread.

**Solution to Exercise 5**

Cross-validation trades $k$ times the training cost for an estimate with an error bar, which is what you need when the validation set is small and a single split could favour either configuration by chance. The two dataset objects with different transforms (Section 2.4) make the "augment the training folds only" requirement a matter of choosing which object each `Subset` points to. Under `FAST_RUN` we use 3 folds of a 3,000-image subset and one epoch; the full run uses 5 folds of 10,000 images and three epochs.

In [ ]:
K_FOLDS = 3 if FAST_RUN else 5
CV_N = 3_000 if FAST_RUN else 10_000
CV_EPOCHS = 1 if FAST_RUN else 3
cv_indices = np.random.default_rng(SEED).permutation(np.array(train_idx[:CV_N]))
folds = np.array_split(cv_indices, K_FOLDS)


def cross_validate(make_model, epochs=CV_EPOCHS, lr=2e-3, weight_decay=0.05):
    accs = []
    for k in range(K_FOLDS):
        val_ids = folds[k].tolist()
        train_ids = np.concatenate([folds[j] for j in range(K_FOLDS) if j != k]).tolist()
        tr_loader = DataLoader(Subset(fm_train_aug, train_ids), batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                               pin_memory=PIN, generator=torch.Generator().manual_seed(SEED + k))
        va_loader = DataLoader(Subset(fm_train_plain, val_ids), batch_size=512, num_workers=0, pin_memory=PIN)
        seed_everything(SEED + k)
        model = make_model().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        h = fit(model, tr_loader, va_loader, opt, epochs, verbose=False)
        accs.append(max(h["val_acc"]))
    return float(np.mean(accs)), float(np.std(accs)), accs


for name, make_model in [("MLP, dropout 0.0", lambda: MLP(hidden_dims=(256, 128), dropout=0.0)),
                         ("MLP, dropout 0.3", lambda: MLP(hidden_dims=(256, 128), dropout=0.3))]:
    mean_acc, std_acc, accs = cross_validate(make_model)
    print(f"{name:18s}: {mean_acc:.4f} +/- {std_acc:.4f}   folds: {[round(a, 3) for a in accs]}")

The standard deviation across folds is the yardstick: if the two means differ by less than about one fold standard deviation, the data cannot tell the configurations apart and you should not pick one on the strength of this experiment. With short budgets dropout tends to look slightly worse (it slows training), and its benefit, if any, appears with longer training on small subsets; the honest conclusion from a `FAST_RUN` is "no significant difference", which is a perfectly good result to write down.

## Summary

- An `nn.Module` registers parameters, buffers and submodules assigned as attributes (or held in `nn.ModuleList`/`nn.ModuleDict`); `state_dict` holds parameters *and* buffers, `parameters()` only the trainable tensors, and a plain Python list of layers is invisible to both. Make architectures configurable from a dictionary and check parameter counts and output shapes with a dummy batch or `torchinfo`.
- `Dataset` is two methods, `__len__` and `__getitem__`; `DataLoader` adds sampling, batching and collation. Write a custom `collate_fn` for variable-size examples, a `WeightedRandomSampler` for imbalance, and split by indices with separate augmented and plain dataset objects so that only the training data is augmented.
- The training loop is three functions: `train_one_epoch` (train mode, finite-loss check, zero-grad/backward/clip/step/schedule), `evaluate` (eval mode, `inference_mode`), and `fit` (epochs, early stopping, best-model checkpoint, logging, callbacks, history). Losses and metrics are accumulated as sums and divided by the number of examples.
- Accuracy hides where a model fails; the confusion matrix, per-class accuracy and a grid of confident errors tell you whether to work on the model, the data or the labels.
- TensorBoard's `SummaryWriter` logs scalars, images, histograms, graphs and hyperparameters; keep one directory per run with the configuration saved next to the results, and always log the learning rate.
- A resume checkpoint holds the model, optimizer and scheduler state dicts, the epoch, the history, the config and every RNG state including the loader's generator; load with `map_location="cpu"`, save `state_dict`s rather than pickled models, and inspect `missing_keys`/`unexpected_keys` when loading partially.
- Seed Python, NumPy and PyTorch, pass a generator to the `DataLoader`, and enable deterministic algorithms only when debugging; hardware, kernels and library versions still limit determinism, so report results over several seeds.
- The complete FashionMNIST project reaches about 90% (MLP) and 92–93% (CNN) test accuracy with augmentation, AdamW, a one-cycle schedule and gradient clipping, and its `run_experiment(config)` function is the template for a training script.
- Debug crashes by reading the shapes in the error message and asserting shapes in `forward`; debug `nan`s with `detect_anomaly`; debug silent failures with the overfit-one-batch test, hooks on activations and gradients, and the checklist.
- Move working code out of notebooks into a small package (`data.py`, `models.py`, `engine.py`, `checkpoint.py`), drive runs with config files and scripts, and keep tests that build each model and overfit one batch.

## Further reading

- PyTorch documentation: `torch.nn.Module` (https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html), `torch.utils.data` (https://docs.pytorch.org/docs/stable/data.html), `torch.utils.tensorboard` (https://docs.pytorch.org/docs/stable/tensorboard.html), and the notes on reproducibility (https://docs.pytorch.org/docs/stable/notes/randomness.html).
- PyTorch tutorials: *Saving and Loading Models* and *Writing Custom Datasets, DataLoaders and Transforms* (https://docs.pytorch.org/tutorials/).
- torchvision documentation: *Transforms v2* (https://docs.pytorch.org/vision/stable/transforms.html) and the datasets reference.
- Karpathy (2019), *A Recipe for Training Neural Networks*, blog post (http://karpathy.github.io/2019/04/25/recipe/): the origin of "overfit one batch" and much of the debugging checklist.
- Goodfellow, Bengio and Courville (2016), *Deep Learning*, Chapter 11 (Practical Methodology). https://www.deeplearningbook.org
- Xiao, Rasul and Vollgraf (2017), *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*, arXiv:1708.07747.
- Zhang, Cissé, Dauphin and Lopez-Paz (2018), *mixup: Beyond Empirical Risk Minimization*, ICLR.
- Sculley et al. (2015), *Hidden Technical Debt in Machine Learning Systems*, NeurIPS: why project structure, configuration and reproducibility matter.
- Zhang, Lipton, Li and Smola, *Dive into Deep Learning*, Chapter 6 (Builders' Guide: modules, parameters, custom layers, file I/O). https://d2l.ai
- TensorBoard documentation (https://www.tensorflow.org/tensorboard) for the dashboards used with `SummaryWriter`.